# Replicate & Analyze a W&B Run

1. Fetch hyperparameters from a W&B run
2. Process a batch of images with the recovered model
3. Plot results using `experiments/plotting`

## 0. Environment setup (Colab)

Mount Google Drive datasets and set required env variables.
Skip this cell if running locally — just `source .env` from the repo root.

In [ ]:
import os
IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    # ---------- edit these paths to match your Drive layout ----------
    DRIVE_ROOT = "/content/drive/MyDrive/thesis"

    os.environ["IMAGE_PATH"]  = f"{DRIVE_ROOT}/chexpert_plus/PNG/PNG"
    os.environ["META_PATH"]   = f"{DRIVE_ROOT}/chexpert_plus/df_chexpert_plus_240401.csv"
    os.environ["MASK_DIR"]    = f"{DRIVE_ROOT}/chexpert_plus/masks/"
    os.environ["CACHE_DIR"]   = f"{DRIVE_ROOT}/torchxrayvision/"
    os.environ["OUTPUT_DIR"]  = f"{DRIVE_ROOT}/outputs/"

    # Real GE scanner GT images (needed by get_priors)
    GT_DIR = f"{DRIVE_ROOT}/data/Processed vs unprocessed real GE scanner"

    # Install the project in editable mode
    # !pip install -e /content/drive/MyDrive/thesis
else:
    # Local: assume .env is already sourced or variables are set
    GT_DIR = os.path.join(
        os.path.dirname(os.path.abspath("")), "data",
        "Processed vs unprocessed real GE scanner",
    )

print("IMAGE_PATH:", os.environ.get("IMAGE_PATH", "<not set>"))
print("CACHE_DIR: ", os.environ.get("CACHE_DIR", "<not set>"))

In [ ]:
import sys

# Add both packages to sys.path so we can import from experiments/ and chest_xray_sim/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
EXPERIMENTS_DIR = os.path.join(REPO_ROOT, "experiments")

for p in [REPO_ROOT, EXPERIMENTS_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Repo root: {REPO_ROOT}")
print(f"Experiments dir: {EXPERIMENTS_DIR}")

## 1. Fetch hyperparameters from W&B

In [ ]:
import wandb

wandb.login()

WANDB_ENTITY = "alamst-kth-royal-institute-of-technology"
WANDB_PROJECT = "individual-gradient-search"

RUN_ID = "yiutl75n"  # <-- replace with your run ID

In [ ]:
api = wandb.Api()
run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/runs/{RUN_ID}")
conf = dict(run.config)
del run, api

# Clean up wandb-specific keys that aren't ExperimentArgs fields
conf.pop("batch_size", None)
conf.pop("sub_batch_size", None)
conf.setdefault("log_samples", 10)
conf.setdefault("gmse_weight", 0.0)

print("Hyperparameters:")
for k, v in sorted(conf.items()):
    print(f"  {k}: {v}")

In [ ]:
from models import ExperimentArgs, default_params

config = ExperimentArgs(**conf)
w0 = default_params(config)

print("Initial weights:", w0)

## 2. Load dataset and run optimization

In [ ]:
from segmentation_utils import get_priors

cache_dir = os.environ["CACHE_DIR"]

# get_priors needs the real GE scanner images to compute anatomical value ranges
gt_tm_path = os.path.join(GT_DIR, "Z01-oprocess.tif")
gt_proc_path = os.path.join(GT_DIR, "Z01-process.tif")

_, value_ranges = get_priors(
    cache_dir,
    collimated_region_bound=0.4,
    real_tm_paths=[gt_tm_path],
    fwd_img_paths=[gt_proc_path],
)
_, priors = get_priors(
    cache_dir,
    real_tm_paths=[gt_tm_path],
    fwd_img_paths=[gt_proc_path],
)

print("Value ranges:", value_ranges)
print("Priors:", priors)

In [ ]:
from chest_xray_sim.data.dataset import Loader
from chest_xray_sim.data.segmentation_dataset import SegmentationChexpertDataset

BATCH_SIZE = 8  # keep small for interactive use
NUM_BATCHES = 1

dataset = SegmentationChexpertDataset(
    root=os.environ["IMAGE_PATH"],
    meta_dir=os.environ["META_PATH"],
    mask_dir=os.environ["MASK_DIR"],
    cache_dir=cache_dir,
    split="train",
    frontal_lateral="Frontal",
)

loader = Loader(
    dataset,
    batch_size=BATCH_SIZE,
    collate_fn=SegmentationChexpertDataset.collate,
    shuffle=False,
)

print(f"Dataset size: {len(dataset)}")
print(f"Processing {NUM_BATCHES} batch(es) of {BATCH_SIZE} images")

In [ ]:
import functools
import jax.numpy as jnp
from replicate import execute

metrics_list, imgs_list, masks_list, txm_list, weights_list, pred_list = execute(
    loader, config, w0, value_ranges, save_dir=None, batches=NUM_BATCHES,
)

# Concatenate across batches
imgs = jnp.concatenate(imgs_list, axis=0)
masks = jnp.concatenate(masks_list, axis=0)
txm = jnp.concatenate(txm_list, axis=0)
pred = jnp.concatenate(pred_list, axis=0)
weights = {
    k: jnp.concatenate([w[k] for w in weights_list], axis=0)
    for k in weights_list[0].keys()
}
metrics = functools.reduce(
    lambda x, y: x.concat(y), metrics_list[1:], metrics_list[0]
) if len(metrics_list) > 1 else metrics_list[0]

print(f"Processed {imgs.shape[0]} images")
print(f"TXM shape: {txm.shape}")
print(f"Pred shape: {pred.shape}")

## 3. Plot results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

from plotting import (
    plot_metric_distributions,
    plot_metrics_boxplot,
    plot_weights_boxplot,
    plot_histogram_heatmap,
    plot_percentile_samples,
    plot_extreme_histograms,
    plot_sample_comparison,
    segmentation_histogram,
)

In [ ]:
fig, ax = plot_metric_distributions(metrics)
plt.show()

In [ ]:
fig, ax = plot_metrics_boxplot(metrics)
plt.show()

In [ ]:
fig, ax = plot_weights_boxplot(weights)
plt.show()

In [ ]:
fig, ax = plot_histogram_heatmap(txm)
plt.show()

In [ ]:
fig, ax = plot_percentile_samples(pred, txm, metrics.ms_ssim)
plt.show()

In [ ]:
fig, ax = plot_extreme_histograms(metrics, txm, masks, priors)
plt.show()

In [ ]:
fig1, fig2 = plot_sample_comparison(imgs, txm, pred, masks, priors)
plt.show()

## 4. Ground truth comparison (optional)

Compare against the real GE scanner transmission map.

In [ ]:
import numpy as np
from chest_xray_sim.data.utils import read_image
from plotting.ground_truth import plot_gt_comparison, plot_gt_vs_naive, plot_gt_metrics

gt_txm = jnp.array(read_image(gt_tm_path)).squeeze()
gt_proc = jnp.array(read_image(gt_proc_path)).squeeze()
naive = (1 - gt_proc) / 2.0

print(f"GT TXM shape: {gt_txm.shape}")
print(f"GT proc shape: {gt_proc.shape}")

In [ ]:
fig, ax = plot_gt_vs_naive(gt_txm, gt_proc, masks[0].squeeze(), priors)
plt.show()

To compare recovered vs GT, run the model on the GT processed image
and then use `plot_gt_comparison` and `plot_gt_metrics`.

This requires running `run_processing` on the single GT image (see `ground_truth_compare.py`).

## Alternative: load from cache

If you already ran `replicate.py` and have cached results, skip the
optimization and load directly:

In [ ]:
# from plotting import load_cache
#
# cache_path = "../replicates_cache/yiutl75n_64_5"  # adjust to your cache
# data = load_cache(cache_path)
#
# imgs = data["imgs"]
# masks = data["masks"]
# txm = data["txm"]
# pred = data["pred"]
# weights = data["weights"]
# metrics = data["metrics"]
#
# # Then use the plotting functions above